In [4]:
import json
from pathlib import Path
from pprint import pprint

PROJECT_ROOT = Path("..")

GOLD_PATH = PROJECT_ROOT / "data" / "processed" / "test.jsonl"
PRED_PATH = PROJECT_ROOT / "predictions" / "rule_baseline.jsonl"

def read_jsonl(path: Path) -> list[dict]:
    rows = []

    with path.open('r', encoding='utf-8') as f:
        for line in f:
            rows.append(json.loads(line))

    return rows

gold_rows = read_jsonl(GOLD_PATH)
pred_rows = read_jsonl(PRED_PATH)

print(len(gold_rows), len(pred_rows))

150 150


In [14]:
errors = []

for idx, (gold_row, pred_row) in enumerate(zip(gold_rows, pred_rows)):
    gold = gold_row["target"]
    pred = pred_row["prediction"]
    text = gold_row["messages"][1]["content"]

    field_errors = []

    for field in ["ticket_type", "topic", "urgency"]:
        if gold[field] != pred[field]:
            field_errors.append(field)

    gold_tags = set(gold["tags"])
    pred_tags = set(pred["tags"])

    if gold_tags != pred_tags:
        field_errors.append("tags")

    if field_errors:
        errors.append({
            "idx": idx,
            "fields": field_errors,
            "text": text,
            "gold": gold,
            "pred": pred,
        })

print(f"Total error rows: {len(errors)} / {len(gold_rows)}")
print(f"Error rate: {len(errors) / len(gold_rows):.4f}")

Total error rows: 150 / 150
Error rate: 1.0000


In [6]:
for err in errors[:5]:
    print("=" * 100)
    print(f"IDX: {err['idx']}")
    print(f"ERROR FIELDS: {err['fields']}")
    print("\nTEXT:")
    print(err["text"][:1000])

    print("\nGOLD:")
    pprint(err["gold"])

    print("\nPRED:")
    pprint(err["pred"])

IDX: 0
ERROR FIELDS: ['urgency', 'tags']

TEXT:
Subject: Payment Details for Billing

Body: Dear Customer Support,

I am reaching out to request a detailed statement of billing payments for my recent acquisitions of hardware and software, including peripheral devices and computers. These items are intended for healthcare procurement, and I require a complete invoice with itemized details to ensure proper accounting and compliance with internal procedures. Furthermore, please include the applicable payment guidelines and relevant terms and conditions related to these purchases. Your prompt assistance would be highly appreciated to facilitate efficient processing.

Thank you.

GOLD:
{'tags': ['Billing',
          'Payment',
          'Invoice',
          'Hardware',
          'Software',
          'Procurement'],
 'ticket_type': 'Request',
 'topic': 'Billing and Payments',
 'urgency': 'high'}

PRED:
{'tags': ['Account', 'Billing', 'Documentation', 'Tech Support'],
 'ticket_type': 'Reques

In [9]:
from collections import Counter

field_counter = Counter()

for err in errors:
    for field in err["fields"]:
        field_counter[field] += 1

for field, count in field_counter.most_common():
    print(f"{field}: {count}")

tags: 150
topic: 98
urgency: 77
ticket_type: 53


In [10]:
topic_errors = [err for err in errors if "topic" in err["fields"]]

for err in topic_errors[:5]:
    print("=" * 100)
    print(f"IDX: {err['idx']}")
    print(err["text"][:700])

    print("\nGOLD topic:", err["gold"]["topic"])
    print("PRED topic:", err["pred"]["topic"])

IDX: 1
Subject: Revise Digital Marketing Approaches

Body: Customer Support, requesting an update on digital marketing strategies to boost brand development and enhance campaign effectiveness. The current tactics are outdated; therefore, a refresh is necessary to maintain competitiveness in the market. The client is interested in exploring new channels, including social media influencer collaborations, to broaden reach and strengthen online visibility. Additionally, they aim to optimize the website for a better user experience.

GOLD topic: Product Support
PRED topic: Technical Support
IDX: 2
Subject: Support for SQL Optimization

Body: I need assistance in optimizing the performance of my SQL Server for data analytics workloads. Could you provide guidance on how to improve query performance and reduce latency? I would greatly appreciate any tips or practices you can share.

GOLD topic: Billing and Payments
PRED topic: Technical Support
IDX: 3
Subject: Service Interruption

Body: Dear 

In [11]:
topic_errors = [err for err in errors if "topic" in err["fields"]]

for err in topic_errors[:5]:
    print("=" * 100)
    print(f"IDX: {err['idx']}")
    print(err["text"][:700])

    print("\nGOLD topic:", err["gold"]["topic"])
    print("PRED topic:", err["pred"]["topic"])

IDX: 1
Subject: Revise Digital Marketing Approaches

Body: Customer Support, requesting an update on digital marketing strategies to boost brand development and enhance campaign effectiveness. The current tactics are outdated; therefore, a refresh is necessary to maintain competitiveness in the market. The client is interested in exploring new channels, including social media influencer collaborations, to broaden reach and strengthen online visibility. Additionally, they aim to optimize the website for a better user experience.

GOLD topic: Product Support
PRED topic: Technical Support
IDX: 2
Subject: Support for SQL Optimization

Body: I need assistance in optimizing the performance of my SQL Server for data analytics workloads. Could you provide guidance on how to improve query performance and reduce latency? I would greatly appreciate any tips or practices you can share.

GOLD topic: Billing and Payments
PRED topic: Technical Support
IDX: 3
Subject: Service Interruption

Body: Dear 

In [12]:
ticket_type_errors = [err for err in errors if "ticket_type" in err["fields"]]

for err in ticket_type_errors[:5]:
    print("=" * 100)
    print(f"IDX: {err['idx']}")
    print(err["text"][:700])

    print("\nGOLD ticket_type:", err["gold"]["ticket_type"])
    print("PRED ticket_type:", err["pred"]["ticket_type"])

IDX: 1
Subject: Revise Digital Marketing Approaches

Body: Customer Support, requesting an update on digital marketing strategies to boost brand development and enhance campaign effectiveness. The current tactics are outdated; therefore, a refresh is necessary to maintain competitiveness in the market. The client is interested in exploring new channels, including social media influencer collaborations, to broaden reach and strengthen online visibility. Additionally, they aim to optimize the website for a better user experience.

GOLD ticket_type: Change
PRED ticket_type: Request
IDX: 3
Subject: Service Interruption

Body: Dear Customer Support,

I am submitting a report concerning a technical failure impacting the Cloud SaaS platform. The service is currently inaccessible, causing significant operational disruptions. Could you please provide an update on the cause of the outage, an estimated time for resolution, and any information on measures to lessen the impact duration? We depend h

In [13]:
tag_errors = [err for err in errors if "tags" in err["fields"]]

for err in tag_errors[:5]:
    print("=" * 100)
    print(f"IDX: {err['idx']}")
    print(err["text"][:700])

    print("\nGOLD tags:", err["gold"]["tags"])
    print("PRED tags:", err["pred"]["tags"])

IDX: 0
Subject: Payment Details for Billing

Body: Dear Customer Support,

I am reaching out to request a detailed statement of billing payments for my recent acquisitions of hardware and software, including peripheral devices and computers. These items are intended for healthcare procurement, and I require a complete invoice with itemized details to ensure proper accounting and compliance with internal procedures. Furthermore, please include the applicable payment guidelines and relevant terms and conditions related to these purchases. Your prompt assistance would be highly appreciated to facilitate efficient processing.

Thank you.

GOLD tags: ['Billing', 'Payment', 'Invoice', 'Hardware', 'Software', 'Procurement']
PRED tags: ['Account', 'Billing', 'Documentation', 'Tech Support']
IDX: 1
Subject: Revise Digital Marketing Approaches

Body: Customer Support, requesting an update on digital marketing strategies to boost brand development and enhance campaign effectiveness. The current t